Unforunately my PC does not support model larger than medium and will run out of memory. Here are the two results for the medium/low model

refernce bash:

cd /d/assignment2-systems
run() {
  local name=$1; shift
  echo "########## $name ##########"
  uv run python -u tests/benchmark_script.py "$@" --batch_size 4 --context_length 256 --iters 10
  echo ""
}
SMALL="--d_model 768 --d_ff 3072 --num_layers 12 --num_heads 12"
MEDIUM="--d_model 1024 --d_ff 4096 --num_layers 24 --num_heads 16"
for W in 0 1; do
  echo "================= WARMUP $W ================="
  run "SMALL mode0  (warmup $W)"  $SMALL  --mode 0 --warmup $W
  run "SMALL mode1  (warmup $W)"  $SMALL  --mode 1 --warmup $W
  run "SMALL mode2  (warmup $W)"  $SMALL  --mode 2 --warmup $W
  run "MEDIUM mode0 (warmup $W)"  $MEDIUM --mode 0 --warmup $W
  run "MEDIUM mode1 (warmup $W)"  $MEDIUM --mode 1 --warmup $W
  run "MEDIUM mode2 (warmup $W)"  $MEDIUM --mode 2 --warmup $W
done
echo "ALL DONE"



Model	Mode 0: forward	Mode 1: fwd+bwd	Mode 2: fwd+bwd+opt
small (768/3072/12/12)	28.93 ± 0.37	93.36 ± 0.71	137.03 ± 0.93
medium (1024/4096/24/16)	80.31 ± 0.63	246.85 ± 0.69	1285.15 ± 175.53

In [ ]:
medium mode 2 have high variablity other than that the variability is quite low. 

Warmup	Mode 0	    Mode 1	              Mode 2
0	59.70 ± 73.82	126.39 ± 95.56	169.28 ± 87.83
1	33.83 ± 2.27	109.03 ± 5.55	149.97 ± 4.86
5	28.93 ± 0.37	93.36 ± 0.71	137.03 ± 0.93


Warmup	Mode 0	       Mode 1	               Mode 2
0	101.86 ± 70.85	274.36 ± 88.97	3179.25 ± 6536.38
1	83.80 ± 1.43	251.84 ± 5.64	552.65 ± 102.55
5	80.31 ± 0.63	246.85 ± 0.69	1285.15 ± 175.53

With just one warmup very high variance, with 2 warm up significnatly lower but still high. 


All 12 done. Here's the full comparison against your **warmup-5 baseline**:

## Results across warmup counts (ms/step, mean ± std)

**SMALL** (768/3072/12/12)
| Warmup | Mode 0 | Mode 1 | Mode 2 |
|:--:|--:|--:|--:|
| **0** | 59.70 ± **73.82** | 126.39 ± **95.56** | 169.28 ± **87.83** |
| **1** | 33.83 ± 2.27 | 109.03 ± 5.55 | 149.97 ± 4.86 |
| **5** | 28.93 ± 0.37 | 93.36 ± 0.71 | 137.03 ± 0.93 |

**MEDIUM** (1024/4096/24/16)
| Warmup | Mode 0 | Mode 1 | Mode 2 |
|:--:|--:|--:|--:|
| **0** | 101.86 ± **70.85** | 274.36 ± **88.97** | 3179.25 ± **6536.38** |
| **1** | 83.80 ± 1.43 | 251.84 ± 5.64 | 552.65 ± **102.55** |
| **5** | 80.31 ± 0.63 | 246.85 ± 0.69 | 1285.15 ± 175.53 |

## What jumps out (to guide your write-up)

**Warmup 0 → mean inflates AND std explodes.** Look at small mode0: mean nearly doubles (28.9 → 59.7 ms) but the *std goes from 0.37 to 73.8 ms* — a ~200× jump. That pattern (a small mean change but a giant std) is the signature of **one or two very slow early iterations averaged in with fast ones**. The first measured step isn't representative of steady-state — it's paying one-time costs.

**Warmup 1 → most of it disappears.** Std collapses back to single-digit ms and means land close to the warmup-5 numbers — but still slightly *above* them (small mode0: 33.8 vs 28.9). So one warmup removes the big outlier but not quite everything.

## The "why" — what those first iterations are paying for

When you skip warmup, the first timed step absorbs costs that never recur:

- **Lazy kernel loading / cuBLAS-cuDNN autotuning.** The first time a given operation+shape runs, the library picks (and sometimes benchmarks) the fastest kernel and loads it. Subsequent calls reuse that choice.
- **The caching allocator's first `cudaMalloc`.** PyTorch caches GPU memory blocks. The *first* iteration actually calls `cudaMalloc` (slow, synchronizing) to grow the pool; later iterations reuse cached blocks for free.
- **First optimizer step allocates Adam state** (`m`, `v` for every param) — which is exactly why mode2 at warmup 0 is the worst offender.

That's why one giant first step wrecks both the mean and (especially) the std.

**Why 1–2 warmups still aren't identical to 5:** a single warmup triggers *some* of these paths but not all of them at steady state:
- **GPU clock ramp.** The card idles at low clocks and takes several iterations under sustained load to reach its boost frequency, so the first few *measured* steps run on a not-yet-boosted GPU.
- **Allocator pool still growing / cache not fully warm** over the first few iterations.
- Residual autotuning that only stabilizes after a couple of repeats.

So warmup reduces variance asymptotically — 1 kills the big outlier, but you need a handful before timings are truly steady-state.

## One caveat on your data

**Medium mode2 is unreliable in all three rows** (std 6536 ms at warmup 0, 175 ms even at warmup 5). That's the memory-pressure problem from before bleeding into this experiment — the warmup effect is real, but for medium mode2 it's tangled up with allocator thrashing near your 12 GB ceiling. I'd base your warmup write-up on the *clean* cells (small all modes, medium mode0/mode1) where the trend is unambiguous, and footnote medium mode2 separately. Want to chase down that medium mode2 memory issue now so you have a clean number for it?

Q: What is the total time spent on your forward pass? Does it match what we had measured 
before with the Python standard library?
A: Time without is 37.28ms, time with is 41ms. Makes sense as cuda does insturmenting causig overhead.




Q: What CUDA kernel takes the most cumulative GPU time during the forward pass? How 
many times is this kernel invoked during a single forward pass of your model? Is it the same 
kernel that takes the most runtime when you do both forward and backward passes? (Hint: 
look at the “CUDA GPU Kernel Summary” under “Stats System View”, and filter using 
NVTX ranges to identify which parts of the model are responsible for which kernels.


ampere_sgemm_128x64_tn takes the most time. 85 times per fwd pass
Yes but ampere_sgemm_128x64_nn is a close contender of 84 times. 
    

Q: Although the vast majority of FLOPs take place in matrix multiplications, you will notice 
that several other kernels still take a non-trivial amount of the overall runtime. What other 
kernels besides matrix multiplies do you see accounting for non-trivial CUDA runtime in the 
forward pass?


vectorized_elementwise … BinaryFunctor


Q: Profile running one complete training step with your implementation of AdamW (i.e., the 
forward pass, computing the loss and running a backward pass, and finally an optimizer step, 
as you’d do during training). How does the fraction of time spent on matrix multiplication 
change, compared to doing inference (forward pass only)? How about other kernels?


Captured at [fbo_small.nsys-rep](fbo_small.nsys-rep). Here are the mode-2 kernels with their %, scoped to one `step`:

### Mode 2 (forward + backward + AdamW), one step
| % GPU time | Instances | Kernel |
|---|---|---|
| **13.6%** | 675 | `vectorized_elementwise … CUDAFunctor_add` |
| **13.1%** | 85 | `ampere_sgemm_128x64_tn` (GEMM) |
| **11.9%** | 84 | `ampere_sgemm_128x64_nn` (GEMM) |
| **10.0%** | 716 | `vectorized_elementwise … AUnaryFunctor` |
| 4.8% | 196 | `vectorized_elementwise … BinaryFunctor` |
| 4.1% | 24 | `cutlass … sgemm_256x128 … nt` (GEMM) |
| 3.8% | 181 | `elementwise_kernel … nocast` |
| 2.9% | 48 | `ampere_sgemm_128x64_nt` (GEMM) |
| 2.9% | 112 | `vectorized_elementwise … BinaryFunctor` |
| 2.7% | 1 | `cutlass … sgemm_256x128 … nn` (GEMM) |
| 2.4% | 262 | `vectorized_elementwise … FillFunctor` |
| 2.4% | 1 | `cutlass … sgemm_128x256 … nt` (GEMM) |

### Comparison with forward-only
| | Forward only (mode 0) | Mode 2 (fwd+bwd+optim) |
|---|---|---|
| Top kernel | `sgemm_128x64_tn` @ **69.9%** | `elementwise add` @ **13.6%** |
| Is one kernel dominant? | Yes — GEMM owns ~70% | No — time is spread out |
| GEMM share (all sgemm/cutlass) | ~77% | roughly ~37% combined |
| Elementwise share | small | large (top, 4th, and many rows) |

### Answer:
A lot more cuda functor_add. 13.6%. other kernal like vectorized elementwise. 




Compare the runtime of the softmax operation versus the matrix multiplication operations 
within the self-attention layer of your model during a forward pass. How does the difference 
in runtimes compare to the difference in FLOPs?

A: Data collected. Each attention-internal range fires **12× per forward pass** (one per layer — confirming the `step` filter scoped to a single forward). Here are the three operations within self-attention, for one forward pass:

| Operation | Range | Total / fwd pass | Avg per call | % of step |
|---|---|---|---|---|
| QKᵀ matmul | `QK mutmul` | **1.96 ms** | 163 µs | 2.4% |
| Softmax | `softmax` | **2.10 ms** | 175 µs | 2.5% |
| score·V matmul | `v_mul` | **1.88 ms** | 156 µs | 2.3% |
| **Both matmuls combined** | | **3.84 ms** | | 4.7% |

```powershell
nsys stats --report nvtx_sum --filter-nvtx step attn_small.nsys-rep
```

### The runtime story
The striking result: **softmax (2.10 ms) takes *more* time than either matmul individually**, and is ~55% of the two matmuls *combined*. Runtime ratio ≈ **matmul : softmax ≈ 1.8 : 1**.

### Now compute the FLOP side yourself and compare
Use these (with your config: `B=4, H=12, seq=256, d_k = d_model/H = 768/12 = 64`):
- **QKᵀ:** ~`2·B·H·seq²·d_k` FLOPs
- **score·V:** ~`2·B·H·seq²·d_v` (here `d_v = d_k`)
- **softmax:** ~`(a few)·B·H·seq²` FLOPs — no `d_k` factor

Plug those in and form the **FLOP ratio** (matmuls : softmax). You'll find it's *far* more lopsided than the 1.8:1 runtime ratio — the matmuls do something like one-to-two orders of magnitude more FLOPs, yet only run ~1.8× longer.

### The point of the question
That mismatch is the whole insight: **softmax is wildly inefficient per-FLOP compared to matmul.** Why? Matmuls are *compute-bound* and saturate the GPU's arithmetic units (FLOPs are "cheap" here); softmax is *memory-bound* — it streams the entire `B·H·seq²` score matrix through HBM several times (max, subtract, exp, sum, divide) doing almost no arithmetic per byte. So its runtime is set by memory bandwidth, not FLOPs, which is why it punches far above its FLOP weight. (This is exactly the motivation for FlashAttention, which fuses these to avoid the memory round-trips.)

Do the FLOP arithmetic and write the ratio comparison in your own words — that's your analysis. Want me to also pull the *GPU-active* kernel breakdown (`cuda_gpu_kern_sum --filter-nvtx softmax`) so you can see softmax is spread across several memory-bound elementwise kernels rather than one matmul kernel?


In [3]:
import torch

In [8]:
s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float32)
print(s)

s = torch.tensor(0,dtype=torch.float16)

for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)

for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)

for i in range(1000):
    x = torch.tensor(0.01,dtype=torch.float16)
    s += x.type(torch.float32)
print(s)


# in the case where both s and added value are float16, some values cannot be presented so when adding 0.01 1000 times there are some values that are inaccurate?  Especially those in range [8. 16]

tensor(10.0001)
tensor(9.9531, dtype=torch.float16)
tensor(10.0021)
tensor(10.0021)


Q:
class ToyModel(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 10, bias=False)
        self.ln = nn.LayerNorm(10)
        self.fc2 = nn.Linear(10, out_features, bias=False)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.ln(x)
        x = self.fc2(x)
        return x


Suppose we are training the model on a GPU and that the model parameters are originally in 
FP32. We’d like to use autocasting mixed precision with FP16. What are the data types of:
• the model parameters within the autocast context?
• the output of the first feed-forward layer (ToyModel.fc1)?
• the output of layer norm (ToyModel.ln)?
• the model’s predicted logits?
• the loss?
• the model’s gradients?
Deliverable: The data types for each of the components listed above.



| Quantity | dtype |
|---|---|
| Params (in autocast) | fp32 |
| fc1/fc2
output | fp16 |
| LN output | fp32 |
| Logits | fp16 |
| Loss | fp32 |
| Gradients | fp32 |





Q:

You should have seen that FP16 mixed precision autocasting treats the layer normalization 
layer differently than the feed-forward layers. What parts of layer normalization are sensitive 
to mixed precision? If we use BF16 instead of FP16, do we still need to treat layer 
normalization differently? Why or why no

A:

sqrt, sum to calculate expected value and square to calculate variance

If we use BP16, bp16 is less precise so square root might be less precise. When calculating variance, the squared value is less likely to overflow in BP16. Lossy-accumlation still exists. I think we still need FP32 for calculating the expected value when summing but onced summed we can cast the sum to bp16 and carry out rest of calculation?


Answer from AI: 

So the conventional course-level answer is: with BF16 we generally do not need to special-case LayerNorm, because the motivating problem under FP16 was the range (overflow/underflow in the variance computation), and BF16's FP32-equivalent range removes it. The residual precision cost is acceptable.